# Video Game Data Integration Pipeline

**Purpose**: Merge critic ratings (IGN), player engagement (Steam), and completion times (HowLongToBeat)

**Output**: Clean dataset for visualization and analysis

## Pipeline Overview
1. **Configuration** - Setup constants and paths
2. **Data Loading** - Load IGN and Steam datasets
3. **Data Cleaning** - Standardize formats and filter data
4. **Data Merging** - Match games across datasets with diagnostics
5. **HLTB Enrichment** - Add completion time data
6. **Final Validation** - Quality checks and justified removals
7. **Export** - Save dataset and metadata

---

## 1. Configuration

In [85]:
# === DATA PROCESSING LIBRARIES ===
import pandas as pd
import numpy as np
from pathlib import Path
import re
from typing import Optional, Dict, Any, List, Tuple

# === API & EXTERNAL LIBRARIES ===
from howlongtobeatpy import HowLongToBeat
import kagglehub

# === UTILITY LIBRARIES ===
import time
import json
from datetime import datetime
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# === FUZZY MATCHING (for improved title matching) ===
try:
    from rapidfuzz import fuzz, process
    FUZZY_MATCHING_AVAILABLE = True
except ImportError:
    print("⚠️ rapidfuzz not available. Install with: pip install rapidfuzz")
    FUZZY_MATCHING_AVAILABLE = False

# === DISPLAY CONFIGURATION ===
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 10)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


In [86]:
# === PIPELINE CONFIGURATION ===

# Matching parameters
FUZZY_MATCH_THRESHOLD = 85  # Minimum similarity score (0-100) for fuzzy title matching
STEAM_LAUNCH_YEAR = 2003  # Steam launched September 2003

# API rate limiting
HLTB_REQUEST_DELAY_SECONDS = 0.5  # Delay between HowLongToBeat API calls
HLTB_MAX_RETRIES = 3  # Maximum retry attempts for failed API calls
HLTB_CHECKPOINT_FREQUENCY = 50  # Save progress every N games

# Data validation thresholds
MIN_EXPECTED_MERGE_RATE = 0.20  # Warn if merge rate below 20%
MAX_PLAYTIME_HOURS = 10000  # Flag games with suspiciously high playtime

# Random seed for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# === PATH CONFIGURATION ===
PROJECT_ROOT = Path.cwd()
CHECKPOINT_DIR = PROJECT_ROOT / 'checkpoints'
OUTPUT_DIR = PROJECT_ROOT / 'output'
LOGS_DIR = PROJECT_ROOT / 'logs'

# Create directories if they don't exist
for directory in [CHECKPOINT_DIR, OUTPUT_DIR, LOGS_DIR]:
    directory.mkdir(exist_ok=True)

# Generate timestamp for this run
RUN_TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')

print(f"📁 Project root: {PROJECT_ROOT}")
print(f"⏰ Run timestamp: {RUN_TIMESTAMP}")
print(f"✅ Configuration complete!")

📁 Project root: /Users/mathusanm6/Code/University/MSc/M2/Visualisation/Critics-vs-Players
⏰ Run timestamp: 20251016_072624
✅ Configuration complete!


In [87]:
# === LOGGING UTILITIES ===

class PipelineLogger:
    """Track data transformations and removals throughout pipeline"""
    
    def __init__(self):
        self.removal_log = []
        self.metrics = {}
        self.stage_counts = {}
    
    def log_removal(self, stage: str, title: str, reason: str, **metadata):
        """Log a data removal with context"""
        self.removal_log.append({
            'stage': stage,
            'title': title,
            'reason': reason,
            'timestamp': datetime.now().isoformat(),
            **metadata
        })
    
    def log_stage(self, stage: str, count: int):
        """Log dataset size at each stage"""
        self.stage_counts[stage] = count
    
    def log_metric(self, name: str, value: Any):
        """Log a pipeline metric"""
        self.metrics[name] = value
    
    def save_logs(self):
        """Save all logs to files"""
        # Save removal log
        if self.removal_log:
            removal_df = pd.DataFrame(self.removal_log)
            removal_path = LOGS_DIR / f'removals_{RUN_TIMESTAMP}.csv'
            removal_df.to_csv(removal_path, index=False)
            print(f"📝 Saved removal log: {removal_path}")
        
        # Save metrics
        metrics_path = LOGS_DIR / f'metrics_{RUN_TIMESTAMP}.json'
        with open(metrics_path, 'w') as f:
            json.dump({
                'stage_counts': self.stage_counts,
                'metrics': self.metrics,
                'timestamp': RUN_TIMESTAMP
            }, f, indent=2)
        print(f"📊 Saved metrics: {metrics_path}")
    
    def print_summary(self):
        """Print pipeline summary"""
        print("\n" + "="*60)
        print("PIPELINE SUMMARY")
        print("="*60)
        print("\nDataset Size by Stage:")
        for stage, count in self.stage_counts.items():
            print(f"  {stage}: {count:,} games")
        
        if self.removal_log:
            print(f"\nTotal Removals: {len(self.removal_log)}")
            removal_df = pd.DataFrame(self.removal_log)
            print("\nRemovals by Stage:")
            print(removal_df['stage'].value_counts().to_string())
            print("\nTop Removal Reasons:")
            print(removal_df['reason'].value_counts().head(5).to_string())

# Initialize logger
logger = PipelineLogger()
print("✅ Logger initialized")

✅ Logger initialized


## 2. Data Loading

In [88]:
# === DOWNLOAD DATASETS ===

print("📥 Downloading datasets from Kaggle...\n")

# Download IGN dataset
print("Downloading IGN Games Dataset (Critics' Ratings)...")
ign_dataset_path = kagglehub.dataset_download("joebeachcapital/ign-games")
print(f"✅ IGN dataset: {ign_dataset_path}\n")

# Download Steam dataset
print("Downloading Steam Games Dataset (Player Engagement)...")
steam_dataset_path = kagglehub.dataset_download("nikdavis/steam-store-games")
print(f"✅ Steam dataset: {steam_dataset_path}\n")

📥 Downloading datasets from Kaggle...

✅ IGN dataset: /Users/mathusanm6/.cache/kagglehub/datasets/joebeachcapital/ign-games/versions/1

✅ Steam dataset: /Users/mathusanm6/.cache/kagglehub/datasets/nikdavis/steam-store-games/versions/3



In [89]:
# === LOAD IGN DATASET ===

ign_csv = list(Path(ign_dataset_path).glob("ign.csv"))[0]
ign_raw = pd.read_csv(ign_csv)

print("📊 IGN DATASET")
print(f"Shape: {ign_raw.shape}")
print(f"Columns: {list(ign_raw.columns)}")
print(f"\nSample:")
display(ign_raw.head(3))

logger.log_stage('ign_raw', len(ign_raw))

📊 IGN DATASET
Shape: (18625, 11)
Columns: ['Unnamed: 0', 'score_phrase', 'title', 'url', 'platform', 'score', 'genre', 'editors_choice', 'release_year', 'release_month', 'release_day']

Sample:


,Unnamed: 0,score_phrase,title,url,platform,score,genre,editors_choice,release_year,release_month,release_day
0,0,Amazing,LittleBigPlanet PS Vita,/games/littlebigplanet-vita/vita-98907,PlayStation Vita,9.0,Platformer,Y,2012,9,12
1,1,Amazing,LittleBigPlanet PS Vita -- Marvel Super Hero E...,/games/littlebigplanet-ps-vita-marvel-super-he...,PlayStation Vita,9.0,Platformer,Y,2012,9,12
2,2,Great,Splice: Tree of Life,/games/splice/ipad-141070,iPad,8.5,Puzzle,N,2012,9,12


In [90]:
# === LOAD STEAM DATASET ===

steam_csv = list(Path(steam_dataset_path).glob("steam.csv"))[0]
steam_raw = pd.read_csv(steam_csv)

print("📊 STEAM DATASET")
print(f"Shape: {steam_raw.shape}")
print(f"Columns: {list(steam_raw.columns)}")
print(f"\nSample:")
display(steam_raw.head(3))

logger.log_stage('steam_raw', len(steam_raw))

📊 STEAM DATASET
Shape: (27075, 18)
Columns: ['appid', 'name', 'release_date', 'english', 'developer', 'publisher', 'platforms', 'required_age', 'categories', 'genres', 'steamspy_tags', 'achievements', 'positive_ratings', 'negative_ratings', 'average_playtime', 'median_playtime', 'owners', 'price']

Sample:


,appid,name,release_date,english,developer,publisher,platforms,required_age,categories,genres,steamspy_tags,achievements,positive_ratings,negative_ratings,average_playtime,median_playtime,owners,price
0,10,Counter-Strike,2000-11-01,1,Valve,Valve,windows;mac;linux,0,Multi-player;Online Multi-Player;Local Multi-P...,Action,Action;FPS;Multiplayer,0,124534,3339,17612,317,10000000-20000000,7.19
1,20,Team Fortress Classic,1999-04-01,1,Valve,Valve,windows;mac;linux,0,Multi-player;Online Multi-Player;Local Multi-P...,Action,Action;FPS;Multiplayer,0,3318,633,277,62,5000000-10000000,3.99
2,30,Day of Defeat,2003-05-01,1,Valve,Valve,windows;mac;linux,0,Multi-player;Valve Anti-Cheat enabled,Action,FPS;World War II;Multiplayer,0,3416,398,187,34,5000000-10000000,3.99


## 3. Data Cleaning

In [91]:
# === TITLE CLEANING UTILITIES ===

def clean_title_level1(title: str) -> str:
    """Level 1: Light cleaning - remove special chars and extra whitespace"""
    if pd.isna(title):
        return ""
    title = str(title)
    # Remove trademark symbols
    title = re.sub(r'[™®©]', '', title)
    # Remove version info in parentheses
    title = re.sub(r'\([^)]*\)', '', title)
    # Normalize whitespace
    title = ' '.join(title.split())
    return title.strip().lower()

def clean_title_level2(title: str) -> str:
    """Level 2: Moderate cleaning - also remove punctuation"""
    title = clean_title_level1(title)
    # Replace -- with - and normalize dashes
    title = title.replace('--', '-')
    # Remove most punctuation except important chars
    title = re.sub(r"[^\w\s\-:'']", '', title)
    return title.strip()

def clean_title_level3(title: str) -> str:
    """Level 3: Aggressive cleaning - remove subtitle after colon/dash"""
    title = clean_title_level2(title)
    # Remove subtitle (everything after first colon or dash)
    title = re.split(r'[:\-]', title)[0].strip()
    return title

# Test cleaning levels
test_title = "Grand Theft Auto V: Premium Edition™ (2015)"
print("Title Cleaning Levels:")
print(f"Original: {test_title}")
print(f"Level 1:  {clean_title_level1(test_title)}")
print(f"Level 2:  {clean_title_level2(test_title)}")
print(f"Level 3:  {clean_title_level3(test_title)}")

Title Cleaning Levels:
Original: Grand Theft Auto V: Premium Edition™ (2015)
Level 1:  grand theft auto v: premium edition
Level 2:  grand theft auto v: premium edition
Level 3:  grand theft auto v


In [92]:
# === CLEAN IGN DATASET ===

print("🧹 Cleaning IGN Dataset...\n")

ign_clean = ign_raw.copy()

# 1. Filter PC platform only
before = len(ign_clean)
ign_clean = ign_clean[ign_clean['platform'] == 'PC']
removed = before - len(ign_clean)
print(f"✓ Filtered to PC platform: removed {removed:,} non-PC games")

# 2. Convert release date
ign_clean['release_date'] = pd.to_datetime({
    'year': ign_clean['release_year'],
    'month': ign_clean['release_month'],
    'day': ign_clean['release_day']
})
print(f"✓ Converted release dates to datetime")

# 3. Filter games after Steam launch
before = len(ign_clean)
ign_clean = ign_clean[ign_clean['release_year'] >= STEAM_LAUNCH_YEAR]
removed = before - len(ign_clean)
print(f"✓ Filtered to post-Steam games (>={STEAM_LAUNCH_YEAR}): removed {removed:,} games")

# 4. Create multiple cleaning levels for matching
ign_clean['title_clean_l1'] = ign_clean['title'].apply(clean_title_level1)
ign_clean['title_clean_l2'] = ign_clean['title'].apply(clean_title_level2)
ign_clean['title_clean_l3'] = ign_clean['title'].apply(clean_title_level3)
print(f"✓ Created multi-level title cleaning")

# 5. Select relevant columns
ign_clean = ign_clean[[
    'title', 'title_clean_l1', 'title_clean_l2', 'title_clean_l3',
    'release_date', 'genre', 'score', 'score_phrase'
]].copy()

# Reset index
ign_clean = ign_clean.reset_index(drop=True)

print(f"\n✅ IGN cleaning complete: {len(ign_clean):,} games")
logger.log_stage('ign_clean', len(ign_clean))

display(ign_clean.head(3))

🧹 Cleaning IGN Dataset...

✓ Filtered to PC platform: removed 15,255 non-PC games
✓ Converted release dates to datetime
✓ Filtered to post-Steam games (>=2003): removed 1,038 games
✓ Created multi-level title cleaning

✅ IGN cleaning complete: 2,332 games


,title,title_clean_l1,title_clean_l2,title_clean_l3,release_date,genre,score,score_phrase
0,Guild Wars 2,guild wars 2,guild wars 2,guild wars 2,2012-09-11,RPG,9.0,Amazing
1,Total War Battles: Shogun,total war battles: shogun,total war battles: shogun,total war battles,2012-09-11,Strategy,7.0,Good
2,Mark of the Ninja,mark of the ninja,mark of the ninja,mark of the ninja,2012-09-07,"Action, Adventure",9.0,Amazing


In [93]:
# === CLEAN STEAM DATASET ===

print("🧹 Cleaning Steam Dataset...\n")

steam_clean = steam_raw.copy()

# 1. Convert release date
steam_clean['release_date'] = pd.to_datetime(steam_clean['release_date'])
print(f"✓ Converted release dates to datetime")

# 2. Calculate owners midpoint
def extract_owners_midpoint(owners_range):
    """Convert owner range string to midpoint integer"""
    try:
        low, high = map(lambda x: int(x.replace(',', '')), owners_range.split('-'))
        return (low + high) // 2
    except:
        return 0

steam_clean['owners_count'] = steam_clean['owners'].apply(extract_owners_midpoint)
print(f"✓ Calculated owner counts from ranges")

# 3. Filter Windows platform
before = len(steam_clean)
steam_clean = steam_clean[steam_clean['platforms'].str.contains('windows', case=False, na=False)]
removed = before - len(steam_clean)
print(f"✓ Filtered to Windows platform: removed {removed:,} non-Windows games")

# 4. Create cleaning levels
steam_clean['title_clean_l1'] = steam_clean['name'].apply(clean_title_level1)
steam_clean['title_clean_l2'] = steam_clean['name'].apply(clean_title_level2)
steam_clean['title_clean_l3'] = steam_clean['name'].apply(clean_title_level3)
print(f"✓ Created multi-level title cleaning")

# 5. Select relevant columns
steam_clean = steam_clean[[
    'name', 'title_clean_l1', 'title_clean_l2', 'title_clean_l3',
    'release_date', 'developer', 'publisher', 'required_age',
    'average_playtime', 'median_playtime', 'owners_count', 'price'
]].copy()

# Reset index
steam_clean = steam_clean.reset_index(drop=True)

print(f"\n✅ Steam cleaning complete: {len(steam_clean):,} games")
logger.log_stage('steam_clean', len(steam_clean))

display(steam_clean.head(3))

🧹 Cleaning Steam Dataset...

✓ Converted release dates to datetime
✓ Calculated owner counts from ranges
✓ Filtered to Windows platform: removed 5 non-Windows games
✓ Created multi-level title cleaning

✅ Steam cleaning complete: 27,070 games


,name,title_clean_l1,title_clean_l2,title_clean_l3,release_date,developer,publisher,required_age,average_playtime,median_playtime,owners_count,price
0,Counter-Strike,counter-strike,counter-strike,counter,2000-11-01,Valve,Valve,0,17612,317,15000000,7.19
1,Team Fortress Classic,team fortress classic,team fortress classic,team fortress classic,1999-04-01,Valve,Valve,0,277,62,7500000,3.99
2,Day of Defeat,day of defeat,day of defeat,day of defeat,2003-05-01,Valve,Valve,0,187,34,7500000,3.99


## 4. Data Merging with Enhanced Matching

In [94]:
# === MULTI-LEVEL MATCHING STRATEGY ===

def merge_with_diagnostics(ign_df, steam_df):
    """
    Merge datasets using tiered matching strategy:
    1. Exact match (Level 1 cleaning)
    2. Moderate match (Level 2 cleaning)
    3. Aggressive match (Level 3 cleaning)
    4. Fuzzy match (if available)
    """
    
    matched_games = []
    match_methods = []
    
    # Create sets for faster lookup
    steam_games_l1 = set(steam_df['title_clean_l1'])
    steam_games_l2 = set(steam_df['title_clean_l2'])
    steam_games_l3 = set(steam_df['title_clean_l3'])
    
    print("🔍 Starting multi-level matching...\n")
    
    for idx, ign_row in tqdm(ign_df.iterrows(), total=len(ign_df), desc="Matching games"):
        match_found = False
        match_method = None
        steam_match = None
        
        # Level 1: Exact match
        if ign_row['title_clean_l1'] in steam_games_l1:
            steam_match = steam_df[steam_df['title_clean_l1'] == ign_row['title_clean_l1']].iloc[0]
            match_method = 'exact_l1'
            match_found = True
        
        # Level 2: Moderate cleaning match
        elif ign_row['title_clean_l2'] in steam_games_l2:
            steam_match = steam_df[steam_df['title_clean_l2'] == ign_row['title_clean_l2']].iloc[0]
            match_method = 'exact_l2'
            match_found = True
        
        # Level 3: Aggressive cleaning match
        elif ign_row['title_clean_l3'] in steam_games_l3:
            steam_match = steam_df[steam_df['title_clean_l3'] == ign_row['title_clean_l3']].iloc[0]
            match_method = 'exact_l3'
            match_found = True
        
        # Level 4: Fuzzy matching (if available)
        elif FUZZY_MATCHING_AVAILABLE:
            best_match = process.extractOne(
                ign_row['title_clean_l2'],
                steam_df['title_clean_l2'].tolist(),
                scorer=fuzz.ratio
            )
            if best_match and best_match[1] >= FUZZY_MATCH_THRESHOLD:
                steam_match = steam_df[steam_df['title_clean_l2'] == best_match[0]].iloc[0]
                match_method = f'fuzzy_{best_match[1]}'
                match_found = True
        
        if match_found:
            # Combine data
            merged_row = {
                'title': ign_row['title'],
                'release_date': ign_row['release_date'],
                'genre': ign_row['genre'],
                'critic_score': ign_row['score'],
                'critic_score_phrase': ign_row['score_phrase'],
                'developer': steam_match['developer'],
                'publisher': steam_match['publisher'],
                'required_age': steam_match['required_age'],
                'average_playtime': steam_match['average_playtime'],
                'median_playtime': steam_match['median_playtime'],
                'owners': steam_match['owners_count'],
                'price': steam_match['price'],
                'match_method': match_method,
            }
            matched_games.append(merged_row)
            match_methods.append(match_method)
    
    merged_df = pd.DataFrame(matched_games)
    
    return merged_df, match_methods

# Perform merge
games_merged, match_methods = merge_with_diagnostics(ign_clean, steam_clean)

🔍 Starting multi-level matching...



Matching games: 100%|██████████| 2332/2332 [00:02<00:00, 1106.17it/s]


In [95]:
# === MERGE DIAGNOSTICS ===

print("\n📊 MERGE DIAGNOSTICS")
print("="*60)

# Overall statistics
merge_rate = len(games_merged) / len(ign_clean) * 100
print(f"\nIGN games: {len(ign_clean):,}")
print(f"Steam games: {len(steam_clean):,}")
print(f"Matched games: {len(games_merged):,}")
print(f"Match rate: {merge_rate:.1f}%")

logger.log_metric('merge_rate', merge_rate)
logger.log_stage('merged', len(games_merged))

# Warn if merge rate is low
if merge_rate < MIN_EXPECTED_MERGE_RATE * 100:
    print(f"\n⚠️  WARNING: Merge rate ({merge_rate:.1f}%) below threshold ({MIN_EXPECTED_MERGE_RATE*100}%)")

# Match method breakdown
print(f"\nMatch Methods:")
method_counts = pd.Series(match_methods).value_counts()
for method, count in method_counts.items():
    pct = count / len(match_methods) * 100
    print(f"  {method}: {count:,} ({pct:.1f}%)")

# Save unmatched games for manual review
matched_titles = set(games_merged['title'].str.lower())
unmatched = ign_clean[~ign_clean['title'].str.lower().isin(matched_titles)]
unmatched_path = LOGS_DIR / f'unmatched_games_{RUN_TIMESTAMP}.csv'
unmatched[['title', 'release_date', 'genre', 'score']].to_csv(unmatched_path, index=False)
print(f"\n📝 Saved {len(unmatched):,} unmatched games to: {unmatched_path}")

display(games_merged.head())


📊 MERGE DIAGNOSTICS

IGN games: 2,332
Steam games: 27,070
Matched games: 1,433
Match rate: 61.4%

Match Methods:
  exact_l1: 819 (57.2%)
  exact_l3: 434 (30.3%)
  exact_l2: 18 (1.3%)
  fuzzy_85.71428571428572: 15 (1.0%)
  fuzzy_88.88888888888889: 9 (0.6%)
  fuzzy_88.0: 8 (0.6%)
  fuzzy_87.5: 7 (0.5%)
  fuzzy_92.3076923076923: 7 (0.5%)
  fuzzy_93.33333333333333: 6 (0.4%)
  fuzzy_95.23809523809523: 6 (0.4%)
  fuzzy_90.9090909090909: 6 (0.4%)
  fuzzy_89.65517241379311: 5 (0.3%)
  fuzzy_90.0: 5 (0.3%)
  fuzzy_89.47368421052632: 4 (0.3%)
  fuzzy_97.87234042553192: 4 (0.3%)
  fuzzy_96.0: 4 (0.3%)
  fuzzy_94.73684210526316: 4 (0.3%)
  fuzzy_89.36170212765957: 4 (0.3%)
  fuzzy_96.2962962962963: 4 (0.3%)
  fuzzy_91.66666666666666: 4 (0.3%)
  fuzzy_86.36363636363636: 3 (0.2%)
  fuzzy_96.7741935483871: 3 (0.2%)
  fuzzy_97.77777777777777: 3 (0.2%)
  fuzzy_97.5609756097561: 2 (0.1%)
  fuzzy_97.2972972972973: 2 (0.1%)
  fuzzy_98.36065573770492: 2 (0.1%)
  fuzzy_98.30508474576271: 2 (0.1%)
  fuzzy_9

,title,release_date,genre,critic_score,critic_score_phrase,developer,publisher,required_age,average_playtime,median_playtime,owners,price,match_method
0,Total War Battles: Shogun,2012-09-11,Strategy,7.0,Good,CREATIVE ASSEMBLY,SEGA,0,177,334,750000,0.00,exact_l3
1,Mark of the Ninja,2012-09-07,"Action, Adventure",9.0,Amazing,Klei Entertainment,Klei Entertainment,16,162,289,75000,15.49,exact_l3
2,Home: A Unique Horror Adventure,2012-09-06,Adventure,6.5,Okay,Benjamin Rivers Inc.,Benjamin Rivers Inc.,0,21,21,350000,1.99,exact_l3
3,Dark Souls (Prepare to Die Edition),2012-08-31,"Action, RPG",9.0,Amazing,QLOC,"FromSoftware, Inc;BANDAI NAMCO Entertainment",0,1160,1417,350000,34.99,exact_l3
4,Symphony,2012-08-30,Shooter,7.0,Good,Empty Clip Studios,Empty Clip Studios,0,321,321,350000,6.19,exact_l1


In [96]:
# === REMOVE DUPLICATE EDITIONS WITH IDENTICAL STEAM METRICS ===

print("🔍 Identifying duplicate editions/DLC with identical Steam metrics...\n")

import re

def extract_base_title(title: str) -> str:
    """Extract base game title by removing subtitles after : or --"""
    if pd.isna(title):
        return ""
    # Split on : or -- and take first part
    base = re.split(r'\s*(?::|--)\s*', str(title))[0].strip()
    return base.lower()

def is_dlc_or_edition(title: str) -> bool:
    """Detect if title is likely DLC or special edition"""
    if pd.isna(title):
        return False
    
    title_lower = str(title).lower()
    
    # DLC indicators
    dlc_keywords = [
        'dlc', 'expansion', 'episode', 'chapter', 'season pass',
        'soundtrack', 'artbook', 'digital deluxe', 'collector',
        'premium edition', 'limited edition', 'special edition',
        'goty', 'game of the year', 'complete edition',
        'enhanced edition', 'definitive edition', 'remastered',
        'director\'s cut', 'workshop', 'pack'
    ]
    
    return any(keyword in title_lower for keyword in dlc_keywords)

# Add base title column
games_merged['base_title'] = games_merged['title'].apply(extract_base_title)

# Group by base title and find duplicates
grouped = games_merged.groupby('base_title')

games_to_remove = []
games_to_keep = []

for base_title, group in grouped:
    if len(group) == 1:
        # No duplicates, keep it
        games_to_keep.extend(group.index.tolist())
        continue
    
    # Check if Steam metrics are identical
    steam_cols = ['average_playtime', 'median_playtime', 'owners']
    
    # Get unique combinations of Steam metrics
    unique_metrics = group[steam_cols].drop_duplicates()
    
    if len(unique_metrics) == 1:
        # ALL entries have identical Steam metrics - keep only one
        # Priority: Keep base game (shortest/simplest title without DLC indicators)
        
        # Score each game (lower is better)
        group_copy = group.copy()
        group_copy['is_dlc'] = group_copy['title'].apply(is_dlc_or_edition)
        group_copy['title_length'] = group_copy['title'].str.len()
        group_copy['has_colon'] = group_copy['title'].str.contains(':').astype(int)
        
        # Sort by: not_dlc (desc), no_colon, shorter title
        group_sorted = group_copy.sort_values(
            by=['is_dlc', 'has_colon', 'title_length'],
            ascending=[True, True, True]
        )
        
        # Keep first (most likely base game), remove rest
        keep_idx = group_sorted.index[0]
        remove_indices = group_sorted.index[1:].tolist()
        
        games_to_keep.append(keep_idx)
        games_to_remove.extend(remove_indices)
        
        # Log removals
        kept_game = games_merged.loc[keep_idx]
        for remove_idx in remove_indices:
            removed_game = games_merged.loc[remove_idx]
            logger.log_removal(
                stage='dlc_deduplication',
                title=removed_game['title'],
                reason=f'Identical Steam metrics to base game "{kept_game["title"]}"',
                base_title=base_title,
                steam_metrics=f"avg={removed_game['average_playtime']}, "
                             f"med={removed_game['median_playtime']}, "
                             f"owners={removed_game['owners']}"
            )
        
        print(f"📦 {base_title}: Kept '{kept_game['title']}', removed {len(remove_indices)} duplicates")
    
    else:
        # Steam metrics differ - keep all
        games_to_keep.extend(group.index.tolist())

# Remove duplicates
before_dedup = len(games_merged)
games_merged = games_merged.loc[games_to_keep].copy()

# Drop temporary column
games_merged = games_merged.drop(columns=['base_title'])

# Reset index
games_merged = games_merged.reset_index(drop=True)

removed_count = before_dedup - len(games_merged)
print(f"\n✅ Deduplication complete:")
print(f"   • Before: {before_dedup:,} games")
print(f"   • After: {len(games_merged):,} games")
print(f"   • Removed: {removed_count:,} duplicate editions/DLC")
print(f"   • Kept all entries with different Steam metrics\n")

logger.log_stage('after_deduplication', len(games_merged))

🔍 Identifying duplicate editions/DLC with identical Steam metrics...

📦 agatha christie: Kept 'Agatha Christie: Death on the Nile', removed 3 duplicates
📦 age of conan: Kept 'Age of Conan: Rise of the Godslayer', removed 1 duplicates
📦 age of empires iii: Kept 'Age of Empires III', removed 2 duplicates
📦 alien shooter: Kept 'Alien Shooter: Vengeance', removed 1 duplicates
📦 anarchy online: Kept 'Anarchy Online: Shadowlands', removed 1 duplicates
📦 assassin's creed: Kept 'Assassin's Creed: Brotherhood', removed 1 duplicates
📦 avatar: Kept 'Avatar: The Game', removed 2 duplicates
📦 battlefield: Kept 'Battlefield: Bad Company 2', removed 2 duplicates
📦 battlestations: Kept 'Battlestations: Midway', removed 1 duplicates
📦 bioshock infinite: Kept 'BioShock Infinite', removed 2 duplicates
📦 blacklight: Kept 'Blacklight: Tango Down', removed 1 duplicates
📦 bone: Kept 'Bone: Out From Boneville', removed 1 duplicates
📦 borderlands: Kept 'Borderlands', removed 4 duplicates
📦 borderlands 2: Kept 

## 5. HowLongToBeat Enrichment

In [97]:
# === HLTB API UTILITIES ===

# Initialize HLTB client
hltb = HowLongToBeat()

# Cache for HLTB results to avoid re-fetching
hltb_cache_path = CHECKPOINT_DIR / 'hltb_cache.json'
hltb_cache = {}

if hltb_cache_path.exists():
    with open(hltb_cache_path, 'r') as f:
        hltb_cache = json.load(f)
    print(f"✅ Loaded {len(hltb_cache)} cached HLTB results")

def search_hltb_with_retry(title: str, max_retries: int = HLTB_MAX_RETRIES) -> Optional[Dict[str, float]]:
    """
    Search HowLongToBeat with caching and retry logic.
    
    Returns dict with: main_story, main_extra, completionist, all_styles (in hours)
    """
    # Check cache first
    cache_key = title.lower().strip()
    if cache_key in hltb_cache:
        return hltb_cache[cache_key]
    
    if not title:
        return None
    
    # Try with original title first
    for attempt in range(max_retries):
        try:
            results = hltb.search(title)
            if results and len(results) > 0:
                game = results[0]
                data = {
                    'main_story': getattr(game, 'main_story', 0),
                    'main_extra': getattr(game, 'main_extra', 0),
                    'completionist': getattr(game, 'completionist', 0),
                    'all_styles': getattr(game, 'all_styles', 0)
                }
                hltb_cache[cache_key] = data
                return data
            
            # Try with cleaned title
            cleaned = clean_title_level2(title)
            if cleaned != title.lower():
                results = hltb.search(cleaned)
                if results and len(results) > 0:
                    game = results[0]
                    data = {
                        'main_story': getattr(game, 'main_story', 0),
                        'main_extra': getattr(game, 'main_extra', 0),
                        'completionist': getattr(game, 'completionist', 0),
                        'all_styles': getattr(game, 'all_styles', 0)
                    }
                    hltb_cache[cache_key] = data
                    return data
            
            return None
            
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)  # Exponential backoff
                continue
            return None
    
    return None

def save_hltb_cache():
    """Save HLTB cache to disk"""
    with open(hltb_cache_path, 'w') as f:
        json.dump(hltb_cache, f, indent=2)

print("✅ HLTB utilities initialized")

✅ Loaded 1324 cached HLTB results
✅ HLTB utilities initialized


In [98]:
# === FETCH HLTB DATA ===

print("⏳ Fetching HowLongToBeat data...\n")
print(f"Total games: {len(games_merged):,}")
print(f"Estimated time: ~{len(games_merged) * HLTB_REQUEST_DELAY_SECONDS / 60:.1f} minutes\n")

# Initialize HLTB columns
hltb_cols = ['main_story', 'main_extra', 'completionist', 'all_styles']
for col in hltb_cols:
    games_merged[col] = np.nan

# Track success
hltb_found = 0
hltb_missing = []

# Process with progress bar
for idx, row in tqdm(games_merged.iterrows(), total=len(games_merged), desc="Fetching HLTB"):
    title = row['title']
    
    # Search HLTB
    hltb_data = search_hltb_with_retry(title)
    
    if hltb_data and hltb_data['all_styles'] > 0:
        for col in hltb_cols:
            games_merged.at[idx, col] = hltb_data[col]
        hltb_found += 1
    else:
        hltb_missing.append({
            'title': title,
            'genre': row['genre'],
            'release_date': row['release_date']
        })
    
    # Save checkpoint
    if (idx + 1) % HLTB_CHECKPOINT_FREQUENCY == 0:
        checkpoint_path = CHECKPOINT_DIR / f'hltb_checkpoint_{idx+1}.parquet'
        games_merged.to_parquet(checkpoint_path)
        save_hltb_cache()
    
    # Rate limiting
    time.sleep(HLTB_REQUEST_DELAY_SECONDS)

# Final cache save
save_hltb_cache()

# Report results
hltb_success_rate = hltb_found / len(games_merged) * 100
print(f"\n✅ HLTB enrichment complete")
print(f"Found: {hltb_found:,}/{len(games_merged):,} ({hltb_success_rate:.1f}%)")
print(f"Missing: {len(hltb_missing):,}")

logger.log_metric('hltb_success_rate', hltb_success_rate)
logger.log_stage('hltb_enriched', len(games_merged))

# Save missing games log
if hltb_missing:
    missing_df = pd.DataFrame(hltb_missing)
    missing_path = LOGS_DIR / f'hltb_missing_{RUN_TIMESTAMP}.csv'
    missing_df.to_csv(missing_path, index=False)
    print(f"📝 Saved missing HLTB data log: {missing_path}")

⏳ Fetching HowLongToBeat data...

Total games: 1,220
Estimated time: ~10.2 minutes



Fetching HLTB: 100%|██████████| 1220/1220 [11:50<00:00,  1.72it/s]


✅ HLTB enrichment complete
Found: 1,107/1,220 (90.7%)
Missing: 113
📝 Saved missing HLTB data log: /Users/mathusanm6/Code/University/MSc/M2/Visualisation/Critics-vs-Players/logs/hltb_missing_20251016_072624.csv


## 6. Final Validation & Justified Removals

In [99]:
# === REMOVE GAMES WITHOUT HLTB DATA ===

print("🔍 Validating HLTB data...\n")

before = len(games_merged)
games_no_hltb = games_merged[games_merged['all_styles'].isna() | (games_merged['all_styles'] == 0)]

# Log each removal
for _, row in games_no_hltb.iterrows():
    logger.log_removal(
        stage='hltb_validation',
        title=row['title'],
        reason='No HLTB completion time data available',
        genre=row['genre'],
        release_date=str(row['release_date'])
    )

# Remove games without HLTB data
games_final = games_merged[games_merged['all_styles'] > 0].copy()

print(f"Removed {before - len(games_final)} games without HLTB data")
print(f"Reason: Cannot analyze completion time relationships without this data\n")

logger.log_stage('after_hltb_filter', len(games_final))

🔍 Validating HLTB data...

Removed 113 games without HLTB data
Reason: Cannot analyze completion time relationships without this data



In [100]:
# === REMOVE DUPLICATE GENRES ===

print("🔍 Cleaning genre data...\n")

def deduplicate_genres(genre_string):
    """Remove duplicate genres while preserving order"""
    if pd.isna(genre_string):
        return genre_string
    
    genres = [g.strip() for g in str(genre_string).split(',')]
    unique_genres = []
    seen = set()
    
    for genre in genres:
        if genre and genre.lower() not in seen:
            unique_genres.append(genre)
            seen.add(genre.lower())
    
    return ','.join(unique_genres) if unique_genres else np.nan

games_final['genre'] = games_final['genre'].apply(deduplicate_genres)
print("✓ Removed duplicate genres")

# Remove games with missing genres
before = len(games_final)
games_missing_genre = games_final[games_final['genre'].isna()]

for _, row in games_missing_genre.iterrows():
    logger.log_removal(
        stage='genre_validation',
        title=row['title'],
        reason='Missing genre classification',
        release_date=str(row['release_date'])
    )

games_final = games_final.dropna(subset=['genre'])
print(f"✓ Removed {before - len(games_final)} games with missing genres\n")

logger.log_stage('after_genre_filter', len(games_final))

🔍 Cleaning genre data...

✓ Removed duplicate genres
✓ Removed 1 games with missing genres



In [101]:
# === DETECT AND FLAG OUTLIERS ===

print("🔍 Detecting outliers...\n")

# Flag suspicious playtime values
suspicious_playtime = games_final[
    (games_final['completionist'] > MAX_PLAYTIME_HOURS) |
    (games_final['all_styles'] > MAX_PLAYTIME_HOURS)
]

if len(suspicious_playtime) > 0:
    print(f"⚠️  Found {len(suspicious_playtime)} games with unusually high playtime:")
    for _, row in suspicious_playtime.iterrows():
        print(f"  - {row['title']}: {row['all_styles']:.0f}h (all styles), {row['completionist']:.0f}h (completionist)")
    print("\nNote: These are typically live service/MMO games. Keeping for analysis.\n")
else:
    print("✓ No suspicious playtime outliers detected\n")

# Check for data quality issues
print("Data Quality Checks:")
print(f"  ✓ Missing critic scores: {games_final['critic_score'].isna().sum()}")
print(f"  ✓ Missing playtime data: {games_final['all_styles'].isna().sum()}")
print(f"  ✓ Missing genres: {games_final['genre'].isna().sum()}")
print(f"  ✓ Negative prices: {(games_final['price'] < 0).sum()}")

🔍 Detecting outliers...

⚠️  Found 1 games with unusually high playtime:
  - Smite: 215h (all styles), 12776h (completionist)

Note: These are typically live service/MMO games. Keeping for analysis.

Data Quality Checks:
  ✓ Missing critic scores: 0
  ✓ Missing playtime data: 0
  ✓ Missing genres: 0
  ✓ Negative prices: 0


In [102]:
# === DROP TEMPORARY COLUMNS ===

# Remove columns used only for matching
columns_to_drop = ['match_method']
games_final = games_final.drop(columns=[c for c in columns_to_drop if c in games_final.columns])

# Reset index
games_final = games_final.reset_index(drop=True)

print(f"\n✅ Final dataset: {len(games_final):,} games")
print(f"Columns: {list(games_final.columns)}\n")

logger.log_stage('final', len(games_final))

display(games_final.head())
display(games_final.describe())


✅ Final dataset: 1,106 games
Columns: ['title', 'release_date', 'genre', 'critic_score', 'critic_score_phrase', 'developer', 'publisher', 'required_age', 'average_playtime', 'median_playtime', 'owners', 'price', 'main_story', 'main_extra', 'completionist', 'all_styles']



,title,release_date,genre,critic_score,critic_score_phrase,developer,publisher,required_age,average_playtime,median_playtime,owners,price,main_story,main_extra,completionist,all_styles
0,1001 Spikes,2014-06-08,Platformer,8.0,Great,"Nicalis, Inc.","Nicalis, Inc.",0,214,215,75000,9.99,7.14,14.37,16.91,13.38
1,140,2013-10-16,Platformer,8.0,Great,Carlsen Games,Carlsen Games,0,205,274,350000,3.99,1.20,2.23,5.26,1.56
2,1979 Revolution,2016-04-21,"Action,Adventure",8.0,Great,iNK Stories;N-Fusion Interactive,iNK Stories,0,0,0,75000,4.79,2.10,2.85,5.88,2.66
3,7 Wonders: Treasures of Seven,2008-11-06,Puzzle,5.9,Mediocre,MumboJumbo,MumboJumbo,0,0,0,35000,6.99,7.10,7.42,9.00,7.43
4,A Bird Story,2014-11-26,RPG,8.8,Great,Freebird Games,Freebird Games,0,257,476,350000,2.79,1.27,1.39,1.41,1.34


,release_date,critic_score,required_age,average_playtime,median_playtime,owners,price,main_story,main_extra,completionist,all_styles
count,1106,1106.000000,1106.000000,1106.000000,1106.000000,1.106000e+03,1106.000000,1106.000000,1106.000000,1106.000000,1106.00000
mean,2010-08-18 05:28:06.075949568,7.514286,1.849005,793.696203,691.024412,1.320886e+06,10.734304,18.610344,35.117025,94.624684,34.08774
min,2003-01-29 00:00:00,1.500000,0.000000,0.000000,0.000000,1.000000e+04,0.000000,0.000000,0.000000,0.000000,0.41000
25%,2007-10-30 12:00:00,6.800000,0.000000,2.000000,2.250000,7.500000e+04,5.640000,5.160000,6.272500,8.532500,7.09000
50%,2011-03-16 12:00:00,7.800000,0.000000,195.500000,198.000000,3.500000e+05,7.990000,9.070000,12.215000,17.570000,11.97500
75%,2013-10-25 00:00:00,8.500000,0.000000,544.250000,463.500000,1.500000e+06,14.990000,16.495000,25.067500,43.047500,25.24750
max,2016-09-22 00:00:00,10.000000,18.000000,95245.000000,190489.000000,1.500000e+08,49.990000,1553.130000,3177.160000,12776.490000,2486.24000
std,NaN,1.371921,5.310945,3544.982191,5932.760496,5.544644e+06,8.050511,62.403649,138.471431,546.155482,121.72024


## 7. Export & Documentation

In [103]:
# === SAVE FINAL DATASET ===

output_csv = OUTPUT_DIR / f'games_final_{RUN_TIMESTAMP}.csv'
output_parquet = OUTPUT_DIR / f'games_final_{RUN_TIMESTAMP}.parquet'

# Save in multiple formats
games_final.to_csv(output_csv, index=False)
games_final.to_parquet(output_parquet, index=False)

print(f"💾 Saved final dataset:")
print(f"  CSV: {output_csv}")
print(f"  Parquet: {output_parquet}")

# Also save as generic output.csv for backwards compatibility
games_final.to_csv(OUTPUT_DIR / 'output.csv', index=False)
print(f"  Latest: {OUTPUT_DIR / 'output.csv'}")

💾 Saved final dataset:
  CSV: /Users/mathusanm6/Code/University/MSc/M2/Visualisation/Critics-vs-Players/output/games_final_20251016_072624.csv
  Parquet: /Users/mathusanm6/Code/University/MSc/M2/Visualisation/Critics-vs-Players/output/games_final_20251016_072624.parquet
  Latest: /Users/mathusanm6/Code/University/MSc/M2/Visualisation/Critics-vs-Players/output/output.csv


In [104]:
# === GENERATE DATA DICTIONARY ===

data_dictionary = {
    'title': 'Game title (from IGN)',
    'release_date': 'Release date (YYYY-MM-DD)',
    'genre': 'Game genre(s), comma-separated',
    'critic_score': 'IGN critic score (0-10 scale)',
    'critic_score_phrase': 'IGN rating category (e.g., "Great", "Amazing")',
    'developer': 'Game developer (from Steam)',
    'publisher': 'Game publisher (from Steam)',
    'required_age': 'Minimum age requirement (from Steam)',
    'average_playtime': 'Average playtime in minutes (Steam user data)',
    'median_playtime': 'Median playtime in minutes (Steam user data)',
    'owners': 'Estimated number of owners (midpoint of Steam range)',
    'price': 'Price in USD (from Steam)',
    'main_story': 'Hours to complete main story (HowLongToBeat)',
    'main_extra': 'Hours to complete main story + extras (HowLongToBeat)',
    'completionist': 'Hours to 100% complete (HowLongToBeat)',
    'all_styles': 'Average hours across all play styles (HowLongToBeat)'
}

dict_df = pd.DataFrame([
    {'column': k, 'description': v, 'dtype': str(games_final[k].dtype)}
    for k, v in data_dictionary.items()
])

dict_path = OUTPUT_DIR / f'data_dictionary_{RUN_TIMESTAMP}.csv'
dict_df.to_csv(dict_path, index=False)
print(f"\n📖 Saved data dictionary: {dict_path}")


📖 Saved data dictionary: /Users/mathusanm6/Code/University/MSc/M2/Visualisation/Critics-vs-Players/output/data_dictionary_20251016_072624.csv


In [105]:
# === GENERATE DATA QUALITY REPORT ===

quality_report = {
    'pipeline_metadata': {
        'run_timestamp': RUN_TIMESTAMP,
        'total_runtime_minutes': None,  # Could track this
        'config': {
            'fuzzy_match_threshold': FUZZY_MATCH_THRESHOLD,
            'STEAM_LAUNCH_YEAR': STEAM_LAUNCH_YEAR
        }
    },
    'data_sources': {
        'ign_raw': len(ign_raw),
        'steam_raw': len(steam_raw)
    },
    'pipeline_stages': logger.stage_counts,
    'quality_metrics': {
        'merge_rate': f"{logger.metrics.get('merge_rate', 0):.2f}%",
        'hltb_success_rate': f"{logger.metrics.get('hltb_success_rate', 0):.2f}%",
        'total_removals': len(logger.removal_log),
        'final_dataset_size': len(games_final)
    },
    'data_completeness': {
        'complete_records': int((games_final.notna().all(axis=1)).sum()),
        'missing_by_column': games_final.isna().sum().to_dict()
    },
    'outliers': {
        'high_playtime_games': len(suspicious_playtime) if 'suspicious_playtime' in locals() else 0
    }
}

report_path = OUTPUT_DIR / f'quality_report_{RUN_TIMESTAMP}.json'
with open(report_path, 'w') as f:
    json.dump(quality_report, f, indent=2, default=str)

print(f"📊 Saved quality report: {report_path}")

📊 Saved quality report: /Users/mathusanm6/Code/University/MSc/M2/Visualisation/Critics-vs-Players/output/quality_report_20251016_072624.json


In [106]:
# === SAVE ALL LOGS ===

logger.save_logs()
logger.print_summary()

📝 Saved removal log: /Users/mathusanm6/Code/University/MSc/M2/Visualisation/Critics-vs-Players/logs/removals_20251016_072624.csv
📊 Saved metrics: /Users/mathusanm6/Code/University/MSc/M2/Visualisation/Critics-vs-Players/logs/metrics_20251016_072624.json

PIPELINE SUMMARY

Dataset Size by Stage:
  ign_raw: 18,625 games
  steam_raw: 27,075 games
  ign_clean: 2,332 games
  steam_clean: 27,070 games
  merged: 1,433 games
  after_deduplication: 1,220 games
  hltb_enriched: 1,220 games
  after_hltb_filter: 1,107 games
  after_genre_filter: 1,106 games
  final: 1,106 games

Total Removals: 327

Removals by Stage:
stage
dlc_deduplication    213
hltb_validation      113
genre_validation       1

Top Removal Reasons:
reason
No HLTB completion time data available                                    113
Identical Steam metrics to base game "Dragon Age: Origins"                  8
Identical Steam metrics to base game "The Sims 3"                           7
Identical Steam metrics to base game "Ever

In [107]:
# === FINAL SUMMARY ===

print("\n" + "="*70)
print(" " * 20 + "PIPELINE COMPLETE")
print("="*70)

print(f"\n📦 Output Files:")
print(f"  • Final dataset: {output_csv.name}")
print(f"  • Data dictionary: data_dictionary_{RUN_TIMESTAMP}.csv")
print(f"  • Quality report: quality_report_{RUN_TIMESTAMP}.json")
print(f"  • Removal log: removals_{RUN_TIMESTAMP}.csv")
print(f"  • Metrics: metrics_{RUN_TIMESTAMP}.json")
print(f"  • Unmatched games: unmatched_games_{RUN_TIMESTAMP}.csv")

print(f"\n📊 Final Dataset Statistics:")
print(f"  • Total games: {len(games_final):,}")
print(f"  • Date range: {games_final['release_date'].min()} to {games_final['release_date'].max()}")
print(f"  • Unique genres: {games_final['genre'].nunique()}")
print(f"  • Avg critic score: {games_final['critic_score'].mean():.2f}/10")
print(f"  • Avg completion time: {games_final['all_styles'].mean():.1f} hours")

print(f"\n✅ Ready for visualization and analysis!")
print("="*70)


                    PIPELINE COMPLETE

📦 Output Files:
  • Final dataset: games_final_20251016_072624.csv
  • Data dictionary: data_dictionary_20251016_072624.csv
  • Quality report: quality_report_20251016_072624.json
  • Removal log: removals_20251016_072624.csv
  • Metrics: metrics_20251016_072624.json
  • Unmatched games: unmatched_games_20251016_072624.csv

📊 Final Dataset Statistics:
  • Total games: 1,106
  • Date range: 2003-01-29 00:00:00 to 2016-09-22 00:00:00
  • Unique genres: 41
  • Avg critic score: 7.51/10
  • Avg completion time: 34.1 hours

✅ Ready for visualization and analysis!
